This guide was written by Bjorge Meulemeester on 2026-03-18

It documents how we augment our activity data from C2 recordings to other recording locations in the barrel cortex

In [48]:
from pathlib import Path
import Interface as I
from getting_started import getting_started_dir

In [49]:
evoked_activity_dir = Path(getting_started_dir) / "example_data" / "functional_constraints" / "evoked_activity"
evoked_activity_recordings_dir = evoked_activity_dir / "recordings_from_C2"
evoked_activity_files = list(evoked_activity_recordings_dir.glob("*.param"))
[str(paramfile.name) for paramfile in evoked_activity_files]
evoked_activity = I.scp.NTParameterSet({})
for evoked in evoked_activity_files: evoked_activity.update(I.scp.build_parameters(evoked))
evoked_activity

NTParameterSet({'L2_B1': NTParameterSet({'distribution': 'PSTH', 'intervals': [[6.0, 7.0], [11.0, 12.0], [19.0, 20.0], [21.0, 22.0], [25.0, 26.0]], 'probabilities': [0.0029, 0.0029, 0.0013, 0.0013, 0.0013]}), 'L2_B2': NTParameterSet({'distribution': 'PSTH', 'intervals': [[4.0, 5.0], [8.0, 9.0], [24.0, 25.0], [27.0, 28.0], [34.0, 35.0], [38.0, 39.0], [39.0, 40.0], [40.0, 41.0], [41.0, 42.0], [42.0, 43.0], [45.0, 46.0]], 'probabilities': [0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013]}), 'L2_B3': NTParameterSet({'distribution': 'PSTH', 'intervals': [[4.0, 5.0], [9.0, 10.0], [25.0, 26.0], [31.0, 32.0], [43.0, 44.0], [49.0, 50.0]], 'probabilities': [0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013]}), 'L2_C1': NTParameterSet({'distribution': 'PSTH', 'intervals': [[3.0, 4.0], [6.0, 7.0], [15.0, 16.0], [37.0, 38.0], [40.0, 41.0], [45.0, 46.0], [46.0, 47.0]], 'probabilities': [0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013]}), 'L2_C2': NTParamete

In [56]:
print(
    Path(evoked_activity_dir / "readme.md").read_text()
)

De Kock, C. P. J., Bruno, R. M., Spors, H., & Sakmann, B. (2007). Layer- and cell-type-specific suprathreshold stimulus representation in rat primary somatosensory cortex. The Journal of Physiology, 581(1), 139–154. https://doi.org/10.1113/jphysiol.2006.124321

# Info about this data

## \*_stim

All files ending with "\*_stim.param" are activity data for all cell types in all locations across the barrel cortex, for varying whisker stimuli.

Important: this activity data was not all actually recorded from these different locations. All acticity data has been recorded in the C2 column, for varying surround
whisker stimuli. We augmented the data by reassigning the same data based on the relative coordinates between recording column and whikser stimulus.
This is explained in more detail below.

## Recordings from C2
All data in this "recordings_from_C2" contains activity data for various cell types in the juvenile Wistar rat Barrel Cortex.
All recordings were made in the C2 column.
Data i

So `"_B2"` does not mean the cell was located in the B2 column. It is located in the C2 column, and the suffix means which whisker was deflected.

In [ ]:
# Assign (row, col) coordinates to every whisker column
COL_TO_RELCO = {
    
    'Alpha': (-1.5, -2),  'A1':    (-2, -1), 'A2':   (-2, 0), 'A3':    (-2, 1),  'A4':   (-2, 2),
    'Beta':  (-0.5, -2),  'B1':    (-1, -1), 'B2':   (-1, 0), 'B3':    (-1, 1),  'B4':   (-1, 2),
    'Gamma': (0.5,  -2),  'C1':    ( 0, -1), 'C2':   ( 0, 0), 'C3':    ( 0, 1),  'C4':   ( 0, 2),
    'Delta': (1.5,  -2),  'D1':    ( 1, -1), 'D2':   ( 1, 0), 'D3':    ( 1, 1),  'D4':   ( 1, 2),
                          'E1':    ( 2, -2), 'E2':   ( 2, 0), 'E3':    ( 2, 1),  'E4':   ( 2, 2),
}

def get_equiv_stimwhisker_for_c2_recording(recording_column, deflected_whisker, constrain_to_3x3=True) -> str:
    # Relative coordinate of the recording column
    vec_c2_to_recording_column = COL_TO_RELCO[recording_column]
    # Relative coordinate of the stimulated whisker
    vec_c2_to_whisker = COL_TO_RELCO[deflected_whisker]

    # Vector from stimulated whisker to recording column
    vec_stimwhisker_to_recording_column = tuple(
        b - a 
        for a, b in zip(vec_c2_to_whisker, vec_c2_to_recording_column)
    )

    if any([abs(e) > 1 for e in vec_stimwhisker_to_recording_column]) and constrain_to_3x3:
        # This recording column is more that one row or arc from the stimulated whisker
        return None

    # Inverse vector: from recording column to stimulated whisker
    vec_recording_column_to_stimwhisker = tuple(
        -e 
        for e in vec_stimwhisker_to_recording_column
    )
    # Move vector from col->whis to C2->other whisk
    # Vectors start at C2, since C2 is (0, 0), so this vector is already what we want
    vec_c2_to_equiv_stimwhisker = vec_recording_column_to_stimwhisker
    # However, we need to adjust in case one of the activity columns or recording columns include greeks
    # These are vertically offset by 0.5. To get C2-positioned equivalent relative columns
    # we round to shift greek columns "up". E.g. alpha to B1 is then considered same as B1 to C2
    vec_c2_to_equiv_stimwhisker = tuple(round(e) for e in vec_recording_column_to_stimwhisker)

    RELCO_TO_COL = {v: k for k, v in COL_TO_RELCO.items()}
    try:
        stimwhisker_rel_to_c2 = RELCO_TO_COL[vec_c2_to_equiv_stimwhisker]
    except KeyError as e: 
        raise ValueError(
            f"Could not find the position {vec_c2_to_equiv_stimwhisker} relative to C2. At this point in the code, this should not happen."
        ) from e
    return stimwhisker_rel_to_c2

# For example
recording_col = "B1"
deflect_whisker = "C2"
equiv_stimwhisker = get_equiv_stimwhisker_for_c2_recording(
    recording_column=recording_col,
    deflected_whisker=deflect_whisker
)

print(
f"Simulating a {deflect_whisker} whisker simulus and recording from {recording_col} has activity",
f"similar to the empirically recorded activity when stimulating {equiv_stimwhisker} (and recording from C2)"
)

Simulating a C2 whisker simulus and recording from B1 has activity similar to the empirically recorded activity when stimulating D3 (and recording from C2)


In [55]:
from config.user.cell_types import EXCITATORY, INHIBITORY

surround_whiskers = ['B1', 'B2', 'B3', 'C1', 'C2', 'C3', 'D1', 'D2', 'D3']

second_surround_whiskers = [
    'A1', 'A2', 'A3', 'A4', 
    'E1', 'E2', 'E3', 'E4',
    'B4', 'C4', 'D4',
    "Alpha", "Beta", "Gamma", "Delta"
    ]

all_whiskers = surround_whiskers + second_surround_whiskers


evoked_activity_dir = Path(getting_started_dir) / "example_data" / "functional_constraints" / "evoked_activity"

for deflect_whisker in all_whiskers:
    print(deflect_whisker)

    inferred_evoked_activity = I.scp.NTParameterSet({})
    for celltype in EXCITATORY+INHIBITORY:
        inferred_evoked_activity[celltype] = {}
        for recording_column in all_whiskers:  # same labels
            celltype_column = f"{celltype}_{recording_column}"
            equiv_whisker = get_equiv_stimwhisker_for_c2_recording(recording_column=recording_column, deflected_whisker=deflect_whisker)
            if equiv_whisker == None:
                continue
            activity_data = evoked_activity.get(f"{celltype}_{equiv_whisker}", None)
            if activity_data != None:
                inferred_evoked_activity[celltype][celltype_column] = activity_data

    outname = evoked_activity_dir / str(deflect_whisker + "_stim.param")
    inferred_evoked_activity.save(outname)

B1
B2
B3
C1
C2
C3
D1
D2
D3
A1
A2
A3
A4
E1
E2
E3
E4
B4
C4
D4
Alpha
Beta
Gamma
Delta
